# Notebook 11: Complete Visualization Gallery
## Project: Indian Air Quality Index (AQI) Comprehensive Analysis
## BTech Final Year Project - Data Science & Machine Learning (8th Semester)

### Objective:
Regenerate and save ALL visualizations from Notebooks 1-10 in high quality (300 DPI) for thesis/report submission.

### Prerequisites:
- Complete Notebooks 1-10
- All processed data files in datasets/ folder

### Run Time: 10-15 minutes

## Step 1: Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['savefig.dpi'] = 300

# Create outputs directory if it doesn't exist
os.makedirs(os.path.join('..', 'outputs'), exist_ok=True)
print('✓ Libraries imported!')
print('✓ Output directory ready!')

## Step 2: Load All Data & Models

In [ ]:
# Load datasets
print('Loading data and models...')
df_clean = pd.read_csv(os.path.join('..', 'datasets', 'city_day_cleaned.csv'), parse_dates=['Datetime'])
X_train = pd.read_csv(os.path.join('..', 'datasets', 'X_train.csv'))
X_test = pd.read_csv(os.path.join('..', 'datasets', 'X_test.csv'))
y_train = pd.read_csv(os.path.join('..', 'datasets', 'y_train_reg.csv')).values.ravel()
y_test = pd.read_csv(os.path.join('..', 'datasets', 'y_test_reg.csv')).values.ravel()

# Load saved models
try:
    final_model = joblib.load(os.path.join('..', 'models', 'final_optimized_model.pkl'))
    print('✓ Final model loaded')
except:
    print('⚠ Final model not found, will skip model-dependent plots')

print(f'✓ Data loaded: {len(df_clean):,} records, {X_train.shape[1]} features')

## Step 3: Notebook 01 Visualizations - Data Quality

In [ ]:
print('Generating Notebook 01 visualizations...\n')

# 3.1: Missing Values Bar Chart
missing_values = df_clean.isna().sum()
missing_df = pd.DataFrame({
    'Missing_Count': missing_values,
    'Missing_Percentage': (missing_values / len(df_clean)) * 100
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)

if len(missing_df) > 0:
    plt.figure(figsize=(14, 7))
    bars = plt.barh(missing_df.index[::-1], missing_df['Missing_Percentage'][::-1], 
                    color='salmon', edgecolor='black', alpha=0.8)
    plt.xlabel('Missing Values (%)', fontsize=12, fontweight='bold')
    plt.ylabel('Columns', fontsize=12, fontweight='bold')
    plt.title('Missing Values by Column (Notebook 01)', fontsize=14, fontweight='bold')
    for bar, value in zip(bars, missing_df['Missing_Percentage'][::-1]):
        plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
                 f'{value:.1f}%', va='center', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join('..', 'outputs', 'NB01_missing_values.png'), bbox_inches='tight')
    plt.close()
    print('✓ Saved: NB01_missing_values.png')

# 3.2: AQI Distribution Histogram & Box Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].hist(df_clean['AQI'].dropna(), bins=100, color='orange', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('AQI Value', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('AQI Distribution (Notebook 01)', fontsize=13, fontweight='bold')

axes[1].boxplot(df_clean['AQI'].dropna(), vert=False)
axes[1].set_xlabel('AQI Value', fontsize=11, fontweight='bold')
axes[1].set_title('AQI Box Plot (Notebook 01)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'NB01_aqi_distribution.png'), bbox_inches='tight')
plt.close()
print('✓ Saved: NB01_aqi_distribution.png')

# 3.3: AQI Bucket Pie Chart
bucket_counts = df_clean['AQI_Bucket'].value_counts()
colors_dict = {'Good': '#22C55E', 'Satisfactory': '#84CC16', 'Moderate': '#F59E0B', 
               'Poor': '#F97316', 'Very Poor': '#EF4444', 'Severe': '#8B0000'}
bucket_colors = [colors_dict.get(b, '#808080') for b in bucket_counts.index]

plt.figure(figsize=(10, 8))
plt.pie(bucket_counts.values, labels=bucket_counts.index, autopct='%.1f%%',
        colors=bucket_colors, startangle=90, explode=[0.02]*len(bucket_counts))
plt.title('AQI Category Distribution (Notebook 01)', fontsize=14, fontweight='bold')
plt.savefig(os.path.join('..', 'outputs', 'NB01_aqi_bucket_distribution.png'), bbox_inches='tight')
plt.close()
print('✓ Saved: NB01_aqi_bucket_distribution.png')

## Step 4: Notebook 02 Visualizations - Advanced EDA

In [ ]:
print('\nGenerating Notebook 02 visualizations...\n')

# Extract time features
df_clean['Year'] = df_clean['Datetime'].dt.year
df_clean['Month'] = df_clean['Datetime'].dt.month
df_clean['Season'] = df_clean['Month'].map({12:'Winter', 1:'Winter', 2:'Winter',
                                             3:'Summer', 4:'Summer', 5:'Summer',
                                             6:'Monsoon', 7:'Monsoon', 8:'Monsoon',
                                             9:'Post-Monsoon', 10:'Post-Monsoon', 11:'Post-Monsoon'})

# 4.1: Pearson Correlation Heatmap
pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'AQI']
corr_matrix = df_clean[pollutants].corr(method='pearson')

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            linewidths=0.5, square=True, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Pearson Correlation Matrix (Notebook 02)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'NB02_pearson_correlation.png'), bbox_inches='tight')
plt.close()
print('✓ Saved: NB02_pearson_correlation.png')

# 4.2: Yearly AQI Trends
yearly_trends = df_clean.groupby('Year').agg(
    AQI_Mean=('AQI', 'mean'),
    Record_Count=('AQI', 'count')
)

fig, ax1 = plt.subplots(figsize=(14, 7))
color1, color2 = 'tab:red', 'tab:blue'
ax1.plot(yearly_trends.index, yearly_trends['AQI_Mean'], marker='o', linewidth=2, color=color1, label='Mean AQI')
ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('AQI', fontsize=12, fontweight='bold', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.bar(yearly_trends.index, yearly_trends['Record_Count'], alpha=0.3, color=color2, label='Record Count')
ax2.set_ylabel('Number of Records', fontsize=12, fontweight='bold', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)
plt.title('Yearly AQI Trends (Notebook 02)', fontsize=14, fontweight='bold')
fig.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'NB02_yearly_trends.png'), bbox_inches='tight')
plt.close()
print('✓ Saved: NB02_yearly_trends.png')

# 4.3: Monthly Seasonal Patterns
monthly_patterns = df_clean.groupby('Month')['AQI'].agg(['mean', 'std']).round(1)
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_patterns.index = month_names

plt.figure(figsize=(14, 7))
plt.plot(month_names, monthly_patterns['mean'], marker='o', linewidth=2.5, markersize=8, color='red')
plt.fill_between(range(12), 
                  monthly_patterns['mean'] - monthly_patterns['std'],
                  monthly_patterns['mean'] + monthly_patterns['std'],
                  alpha=0.2, color='red')
plt.axhline(df_clean['AQI'].mean(), color='green', linestyle='--', linewidth=2, 
            label=f'Annual Average: {df_clean["AQI"].mean():.0f}')
plt.xlabel('Month', fontsize=12, fontweight='bold')
plt.ylabel('Average AQI', fontsize=12, fontweight='bold')
plt.title('Monthly Seasonal AQI Pattern (Notebook 02)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'NB02_monthly_patterns.png'), bbox_inches='tight')
plt.close()
print('✓ Saved: NB02_monthly_patterns.png')

# 4.4: Seasonal Box Plot
season_order = ['Winter', 'Summer', 'Monsoon', 'Post-Monsoon']
plt.figure(figsize=(12, 7))
sns.boxplot(data=df_clean, x='Season', y='AQI', order=season_order, palette='viridis')
plt.xlabel('Season', fontsize=12, fontweight='bold')
plt.ylabel('AQI', fontsize=12, fontweight='bold')
plt.title('Seasonal AQI Distribution (Notebook 02)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'NB02_seasonal_boxplot.png'), bbox_inches='tight')
plt.close()
print('✓ Saved: NB02_seasonal_boxplot.png')

## Step 5: Notebook 03 Visualizations - Time Series

In [ ]:
print('\nGenerating Notebook 03 visualizations...\n')

# 5.1: Daily AQI Time Series with Rolling Average
daily_aqi = df_clean.groupby('Datetime')['AQI'].mean().resample('D').mean()
daily_aqi = daily_aqi.interpolate(method='linear')

plt.figure(figsize=(16, 6))
plt.plot(daily_aqi.index, daily_aqi.values, linewidth=0.5, alpha=0.7, color='blue')
rolling_mean = daily_aqi.rolling(window=30).mean()
plt.plot(rolling_mean.index, rolling_mean.values, linewidth=2, color='red', label='30-Day Moving Average')
plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel('Average AQI', fontsize=12, fontweight='bold')
plt.title('Daily AQI Time Series (Notebook 03)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'NB03_daily_aqi_timeseries.png'), bbox_inches='tight')
plt.close()
print('✓ Saved: NB03_daily_aqi_timeseries.png')

# 5.2: Monthly AQI Heatmap by Year
monthly_pivot = df_clean.pivot_table(values='AQI', index='Year', columns='Month', aggfunc='mean')
monthly_pivot.columns = month_names

plt.figure(figsize=(14, 8))
sns.heatmap(monthly_pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5)
plt.title('Monthly AQI Heatmap by Year (Notebook 03)', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12, fontweight='bold')
plt.ylabel('Year', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'NB03_monthly_heatmap.png'), bbox_inches='tight')
plt.close()
print('✓ Saved: NB03_monthly_heatmap.png')

## Step 6: Notebook 10 Visualizations - ML & Ensemble Models

In [ ]:
print('\nGenerating Notebook 10 visualizations...\n')

# 6.1: Feature Importance (from final model)
if 'final_model' in dir():
    try:
        importances = final_model.feature_importances_
        feat_imp_df = pd.DataFrame({
            'Feature': X_train.columns,
            'Importance': importances
        }).sort_values('Importance', ascending=False)

        plt.figure(figsize=(10, 8))
        top_n = min(15, len(feat_imp_df))
        sns.barplot(data=feat_imp_df.head(top_n), x='Importance', y='Feature', 
                    palette='viridis', edgecolor='black')
        plt.xlabel('Feature Importance', fontsize=12, fontweight='bold')
        plt.ylabel('Feature', fontsize=12, fontweight='bold')
        plt.title(f'Top {top_n} Features - Final Optimized Model (Notebook 10)', 
                  fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join('..', 'outputs', 'NB10_feature_importance.png'), bbox_inches='tight')
        plt.close()
        print('✓ Saved: NB10_feature_importance.png')
    except Exception as e:
        print(f'⚠ Could not generate feature importance plot: {e}')
else:
    print('⚠ Final model not loaded, skipping feature importance')

## Step 7: Final Summary

In [ ]:
print('\n' + '='*70)
print('ALL VISUALIZATIONS COMPLETE!')
print('='*70)
print(f'\nTotal images saved to outputs/ folder:')
output_files = [f for f in os.listdir(os.path.join('..', 'outputs')) if f.endswith('.png')]
for i, f in enumerate(sorted(output_files), 1):
    print(f'  {i}. {f}')
print(f'\n✓ Total: {len(output_files)} high-quality images (300 DPI)')
print('✓ Ready for thesis/report submission!')

## Summary

### All Saved Visualizations:

**Notebook 01 - Data Quality:**
1. NB01_missing_values.png
2. NB01_aqi_distribution.png
3. NB01_aqi_bucket_distribution.png

**Notebook 02 - Advanced EDA:**
4. NB02_pearson_correlation.png
5. NB02_yearly_trends.png
6. NB02_monthly_patterns.png
7. NB02_seasonal_boxplot.png

**Notebook 03 - Time Series:**
8. NB03_daily_aqi_timeseries.png
9. NB03_monthly_heatmap.png

**Notebook 10 - ML & Ensemble:**
10. NB10_feature_importance.png

**All images are:**
- ✅ High resolution (300 DPI)
- ✅ Publication quality
- ✅ Properly labeled
- ✅ Ready for thesis/report